# Notebook purpose and notes\n
- Purpose: train DimeNet++ and SchNet with original workflow structure.\n
- Notes: paths and configurable hyperparameters are loaded from ../configs/config.json.\n

In [ ]:
import os
import gc
import csv
import json
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Headless/cloud mode: avoid interactive plot blocking
import matplotlib.pyplot as plt
from datetime import datetime
from collections import Counter
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch_geometric.data import InMemoryDataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import DimeNetPlusPlus, SchNet, global_max_pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# ================= 1. Basic configuration and system setup =================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Current runtime device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
    torch.backends.cudnn.benchmark = True


In [ ]:
# ================= 2. Path Configuration & Experiment Hyperparameters =================
# Compatible when notebook is launched from workspace/ or workspace/scripts/
config_candidates = [
    os.path.join('configs', 'config.json'),
    os.path.join('..', 'configs', 'config.json'),
]
config_path = next((p for p in config_candidates if os.path.exists(p)), None)
if config_path is None:
    raise FileNotFoundError(f'config.json not found in candidates: {config_candidates}')

with open(config_path, 'r', encoding='utf-8') as f:
    CFG = json.load(f)

BASE_PATH = CFG['paths']['data_root']
XYZ_DIR = CFG['paths']['xyz_dir']
LABEL_FILE = CFG['paths']['label_csv']
DATASET_ROOT = os.path.join(BASE_PATH, 'dataset_0214_split')

start_time_str = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = os.path.join(CFG['paths']['output_root'], f"run_combined_updated_{start_time_str}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f">>> [Experiment Initialization] Results directory: {OUTPUT_DIR}")

# ==================== Shared Training Hyperparameters ====================
HYPER_PARAMS = {
    'test_ratio': CFG['train']['test_ratio'],
    'val_ratio': CFG['train']['val_ratio'],
    'batch_size': CFG['train']['batch_size'],
    'learning_rate': CFG['train']['learning_rate'],
    'epochs': CFG['train']['max_epochs'],
    'num_classes': 2,
    'seed': CFG['train']['seed'],
    'optimizer': {'weight_decay': CFG['train']['weight_decay']},
    'hardware': {'num_workers': CFG['train']['num_workers'], 'pin_memory': CFG['train']['pin_memory']}
}

# ==================== DimeNet++ Model Parameters ====================
MODEL_CONFIG_DIMENET = {
    'hidden_channels': CFG['dimenet_model']['hidden_channels'],
    'out_channels': CFG['dimenet_model']['out_channels'],
    'num_blocks': CFG['dimenet_model']['num_blocks'],
    'int_emb_size': CFG['dimenet_model']['int_emb_size'],
    'basis_emb_size': CFG['dimenet_model']['basis_emb_size'],
    'out_emb_channels': CFG['dimenet_model']['out_emb_channels'],
    'num_spherical': CFG['dimenet_model']['num_spherical'],
    'num_radial': CFG['dimenet_model']['num_radial'],
    'cutoff': CFG['dimenet_model']['cutoff'],
    'envelope_exponent': CFG['dimenet_model']['envelope_exponent']
}

# ==================== SchNet Model Parameters ====================
MODEL_CONFIG_SCHNET = {
    'hidden_channels': CFG['schnet_model']['hidden_channels'],
    'num_filters': CFG['schnet_model']['num_filters'],
    'num_interactions': CFG['schnet_model']['num_interactions'],
    'num_gaussians': CFG['schnet_model']['num_gaussians'],
    'cutoff': CFG['schnet_model']['cutoff'],
    'readout': CFG['schnet_model']['readout']
}


In [ ]:

# ================= 3. Dataset Definition & Processing =================
ATOM_MAP = {
    'H': 1, 'He': 2, 'Li': 3, 'Be': 4, 'B': 5, 'C': 6, 'N': 7, 'O': 8, 'F': 9, 'Ne': 10,
    'Na': 11, 'Mg': 12, 'Al': 13, 'Si': 14, 'P': 15, 'S': 16, 'Cl': 17, 'Ar': 18,
    'K': 19, 'Ca': 20, 'Sc': 21, 'Ti': 22, 'V': 23, 'Cr': 24, 'Mn': 25, 'Fe': 26,
    'Co': 27, 'Ni': 28, 'Cu': 29, 'Zn': 30, 'Ga': 31, 'Ge': 32, 'As': 33, 'Se': 34,
    'Br': 35, 'Kr': 36, 'Rb': 37, 'Sr': 38, 'Y': 39, 'Zr': 40, 'Nb': 41, 'Mo': 42,
    'Ru': 44, 'Rh': 45, 'Pd': 46, 'Ag': 47, 'Cd': 48, 'In': 49, 'Sn': 50, 'Sb': 51,
    'Te': 52, 'I': 53, 'Xe': 54, 'Cs': 55, 'Ba': 56, 'La': 57, 'Ce': 58, 'Pr': 59,
    'Nd': 60, 'Pm': 61, 'Sm': 62, 'Eu': 63, 'Gd': 64, 'Tb': 65, 'Dy': 66, 'Ho': 67,
    'Er': 68, 'Tm': 69, 'Yb': 70, 'Lu': 71, 'Hf': 72, 'Ta': 73, 'W': 74, 'Re': 75,
    'Os': 76, 'Ir': 77, 'Pt': 78, 'Au': 79, 'Hg': 80
}

class CageDataset(InMemoryDataset):
    def __init__(self, root, excel_file, xyz_dir, transform=None, pre_transform=None):
        self.excel_file = excel_file
        self.xyz_dir = xyz_dir
        super().__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)

    @property
    def raw_file_names(self):
        return []

    @property
    def processed_file_names(self):
        return ['processed_data.pt']

    def process(self):
        print(f">>> [Dataset] Start processing data...")
        df = pd.read_csv(self.excel_file, header=None)
        if len(df.columns) >= 2:
            df = df.iloc[:, :2]
            df.columns = ['Refcode', 'Label']

        data_list = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Parsing Molecules"):
            refcode = str(row['Refcode']).strip()
            try:
                raw_label = int(row['Label'])
            except:
                continue

            # Binary training mapping: 0->0, 1->1, 2->1
            label = 1 if raw_label in (1, 2) else 0

            filepath = os.path.join(self.xyz_dir, f"{refcode}.xyz")
            if not os.path.exists(filepath):
                continue

            try:
                with open(filepath, 'r') as f:
                    lines = f.readlines()
                    if len(lines) < 3:
                        continue
                    pos, z = [], []
                    for line in lines[2:]:
                        parts = line.split()
                        if len(parts) < 4:
                            continue
                        if parts[0] in ATOM_MAP:
                            z.append(ATOM_MAP[parts[0]])
                            pos.append([float(parts[1]), float(parts[2]), float(parts[3])])
                    if not z:
                        continue
                    data = Data(
                        z=torch.tensor(z, dtype=torch.long),
                        pos=torch.tensor(pos, dtype=torch.float),
                        y=torch.tensor([label], dtype=torch.long),
                    )
                    data.refcode = refcode
                    data_list.append(data)
            except Exception:
                pass

        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

dataset = CageDataset(root=DATASET_ROOT, excel_file=LABEL_FILE, xyz_dir=XYZ_DIR)
print(f"Dataset loaded successfully, total: {len(dataset)}")


In [ ]:
# ================= 4. Dataset Splitting (75% Train, 10% Val, 15% Test) =================
print(f"\n>>> [Dataset Splitting] Ratio: Train 75% | Val 10% | Test 15% (Seed: {HYPER_PARAMS['seed']})")
all_labels = [data.y.item() for data in dataset]
all_indices = list(range(len(dataset)))

train_val_idx, test_idx = train_test_split(
    all_indices, test_size=HYPER_PARAMS['test_ratio'], 
    random_state=HYPER_PARAMS['seed'], stratify=all_labels
)

val_relative_ratio = HYPER_PARAMS['val_ratio'] / (1.0 - HYPER_PARAMS['test_ratio'])
train_val_labels = [all_labels[i] for i in train_val_idx]

train_idx, val_idx = train_test_split(
    train_val_idx, test_size=val_relative_ratio, 
    random_state=HYPER_PARAMS['seed'], stratify=train_val_labels
)

train_dataset = dataset[train_idx]
val_dataset = dataset[val_idx]
test_dataset = dataset[test_idx]

print(f"  - Train set size: {len(train_dataset)} | Distribution: {Counter([data.y.item() for data in train_dataset])}")
print(f"  - Validation set size: {len(val_dataset)} | Distribution: {Counter([data.y.item() for data in val_dataset])}")
print(f"  - Test set size: {len(test_dataset)} | Distribution: {Counter([data.y.item() for data in test_dataset])}")

train_loader = DataLoader(train_dataset, batch_size=HYPER_PARAMS['batch_size'], shuffle=True, 
                          num_workers=HYPER_PARAMS['hardware']['num_workers'], pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=HYPER_PARAMS['batch_size'], shuffle=False, 
                        num_workers=HYPER_PARAMS['hardware']['num_workers'], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=HYPER_PARAMS['batch_size'], shuffle=False, 
                         num_workers=HYPER_PARAMS['hardware']['num_workers'], pin_memory=True)

In [ ]:
# ================= 5. Model Definition =================
def get_dimenet_model():
    model = DimeNetPlusPlus(**MODEL_CONFIG_DIMENET)
    return model.to(DEVICE)

class SchNetClassifier(nn.Module):
    def __init__(self, config, num_classes):
        super().__init__()
        self.schnet = SchNet(
            hidden_channels=config['hidden_channels'],
            num_filters=config['num_filters'],
            num_interactions=config['num_interactions'],
            num_gaussians=config['num_gaussians'],
            cutoff=config['cutoff'],
            readout=config['readout'] 
        )
        in_dim = config['hidden_channels']
        self.classifier = nn.Sequential(
            nn.Linear(in_dim, in_dim // 2),
            nn.BatchNorm1d(in_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2), 
            nn.Linear(in_dim // 2, num_classes)
        )
    def forward(self, z, pos, batch):
        h = self.schnet.embedding(z)
        edge_index, edge_weight = self.schnet.interaction_graph(pos, batch)
        for interaction in self.schnet.interactions:
            h = h + interaction(h, edge_index, edge_weight, self.schnet.distance_expansion(edge_weight))
        graph_emb = global_max_pool(h, batch) 
        out = self.classifier(graph_emb)      
        return out

def get_schnet_model():
    return SchNetClassifier(MODEL_CONFIG_SCHNET, HYPER_PARAMS['num_classes']).to(DEVICE)

In [ ]:
# ================= 6. Universal Training & Validation Function =================
def train_model(model_name, model, train_loader, val_loader, epochs):
    print(f"\n================ Start Training {model_name} ================")
    optimizer = torch.optim.Adam(model.parameters(), lr=HYPER_PARAMS['learning_rate'], 
                                 weight_decay=HYPER_PARAMS['optimizer']['weight_decay'])
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler('cuda')
    
    best_val_acc = 0.0
    safe_model_name = model_name.replace('++', 'plus') # Prevent special characters in filename
    best_model_path = os.path.join(OUTPUT_DIR, f"best_{safe_model_name}.pth")
    log_csv_path = os.path.join(OUTPUT_DIR, f"step_log_{safe_model_name}.csv")
    
    # history used for plotting
    history = {
        'train_steps': [], 'train_loss': [], 'train_acc': [],
        'val_steps': [], 'val_loss': [], 'val_acc': []
    }
    
    # log_records used to export complete CSV table
    log_records = []
    global_step = 0
    
    for epoch in range(1, epochs + 1):
        model.train()
        pbar = tqdm(train_loader, desc=f"[{model_name}] Ep {epoch}/{epochs}", leave=False, ncols=100)
        
        # === A. Training Loop (recorded per batch) ===
        for data in pbar:
            global_step += 1
            data = data.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            targets = data.y.view(-1).long()
            
            with autocast('cuda'): 
                out = model(data.z, data.pos, data.batch)
                loss = criterion(out, targets)
            
            if torch.isnan(loss): continue
            
            scaler.scale(loss).backward()       
            scaler.unscale_(optimizer)          
            scaler.step(optimizer)              
            scaler.update()
            
            batch_loss = loss.item()
            batch_correct = (out.argmax(dim=1) == targets).sum().item()
            batch_total = targets.size(0)
            batch_acc = batch_correct / batch_total if batch_total > 0 else 0
            
            # 1. Record for plotting
            history['train_steps'].append(global_step)
            history['train_loss'].append(batch_loss)
            history['train_acc'].append(batch_acc)
            
            # 2. Record for CSV output (Train)
            log_records.append({
                "Global_Step": global_step,
                "Epoch": epoch,
                "Phase": "Train",
                "Loss": round(batch_loss, 6),
                "Accuracy": round(batch_acc, 4)
            })
            
            pbar.set_postfix({'Loss': f"{batch_loss:.4f}", 'Acc': f"{batch_acc:.4f}"})
            
        # === B. Validation Loop (executed after each epoch) ===
        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(DEVICE, non_blocking=True)
                targets = data.y.view(-1).long()
                with autocast('cuda'):
                    out = model(data.z, data.pos, data.batch)
                    loss = criterion(out, targets)
                v_loss += loss.item()
                v_correct += (out.argmax(dim=1) == targets).sum().item()
                v_total += targets.size(0)
        
        val_loss = v_loss / len(val_loader)
        val_acc = v_correct / v_total if v_total > 0 else 0
        
        # 1. Record for plotting (X-axis aligned to current global_step)
        history['val_steps'].append(global_step) 
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # 2. Record for CSV output (Validation)
        log_records.append({
            "Global_Step": global_step,
            "Epoch": epoch,
            "Phase": "Validation",
            "Loss": round(val_loss, 6),
            "Accuracy": round(val_acc, 4)
        })
        
        # Save the best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  Ep {epoch:03d} * Best model saved (Val Acc: {val_acc:.4f})")
        
        gc.collect()
        torch.cuda.empty_cache()

    # === C. Export complete log to CSV ===
    df_log = pd.DataFrame(log_records)
    df_log.to_csv(log_csv_path, index=False)
    print(f"\n[{model_name}] Step-level training log exported to: {log_csv_path}")

    return history, best_model_path

In [ ]:
# ================= 7. Execute Training and Plotting Sequentially =================
models = {
    "DimeNet++": get_dimenet_model(),
    "SchNet": get_schnet_model()
}

results_history = {}
best_model_paths = {}

for name, model_instance in models.items():
    hist, path = train_model(name, model_instance, train_loader, val_loader, HYPER_PARAMS['epochs'])
    results_history[name] = hist
    best_model_paths[name] = path

# --- Plot Loss and Acc curves based on Batch ---
print("\n>>> Start plotting training curves (X-axis unit: Batch Steps)...")
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Training Metrics per Batch Step', fontsize=16)

for i, (name, hist) in enumerate(results_history.items()):
    # Plot Loss
    ax_loss = axes[0, i]
    ax_loss.plot(hist['train_steps'], hist['train_loss'], color='blue', alpha=0.3, label='Train Loss (per batch)')
    ax_loss.plot(hist['val_steps'], hist['val_loss'], color='red', marker='o', linestyle='-', linewidth=2, label='Val Loss')
    ax_loss.set_title(f'{name} - Loss')
    ax_loss.set_xlabel('Batch Steps')
    ax_loss.set_ylabel('Loss')
    ax_loss.legend()
    
    # Plot Acc
    ax_acc = axes[1, i]
    ax_acc.plot(hist['train_steps'], hist['train_acc'], color='green', alpha=0.3, label='Train Acc (per batch)')
    ax_acc.plot(hist['val_steps'], hist['val_acc'], color='orange', marker='s', linestyle='-', linewidth=2, label='Val Acc')
    ax_acc.set_title(f'{name} - Accuracy')
    ax_acc.set_xlabel('Batch Steps')
    ax_acc.set_ylabel('Accuracy')
    ax_acc.legend()

plt.tight_layout()
PLOT_PATH = os.path.join(OUTPUT_DIR, "combined_training_curves_per_batch.png")
plt.savefig(PLOT_PATH, dpi=300)
print(f"Curve image saved to: {PLOT_PATH}")
print(f"All CSV training logs and model weights have been saved in the directory: {OUTPUT_DIR}")
plt.close(fig)